<div style="text-align:center; border-radius:20px; padding:25px; color:white; margin:0; font-family:sans-serif; background:#1b002a; box-shadow:0px 6px 18px rgba(0,0,0,0.35); margin-bottom:1em;">
          <div style="font-size:200%; color:#FEE100; font-weight:700;">
            2026-fifa-world-cup-historical-elo-ratings-ETL,EDA,Visualize,Model
          </div>
        </div>
        

In [1]:
import os
main_csv_local_path = 'elo_ratings_wc2026.csv'
DATA_DIR = ''
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        if filename == main_csv_local_path:
            DATA_DIR = os.path.join(dirname, filename)
            print(DATA_DIR)


/kaggle/input/datasets/afonsofernandescruz/2026-fifa-world-cup-historical-elo-ratings/elo_ratings_wc2026.csv


In [2]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

# A) Imports + Global Config
warnings.filterwarnings('ignore')
np.random.seed(42)
pd.set_option('display.max_columns', 100)
pd.set_option('display.width', 1000)
plt.rcParams['axes.titlesize'] = 14
plt.rcParams['axes.labelsize'] = 12

DATASET_NAME = "afonsofernandescruz/2026-fifa-world-cup-historical-elo-ratings"
EXACT_COLUMNS = [
    'year', 'snapshot_date', 'country', 'rank', 'country_code', 'rating', 
    'rank_max', 'rating_max', 'rank_avg', 'rating_avg', 'rank_min', 'rating_min', 
    'matches_total', 'matches_home', 'matches_away', 'matches_neutral', 
    'wins', 'losses', 'draws', 'goals_for', 'goals_against', 'confederation', 'is_host'
]
TARGET_COL = None

# I) Engineering Requirements - helper functions
def safe_read_csv(path):
    try:
        df = pd.read_csv(path)
        return df
    except Exception as e:
        print(f"Error reading CSV: {e}")
        return pd.DataFrame()

def audit_missingness(df):
    try:
        null_counts = df.isnull().sum()
        null_pcts = (null_counts / len(df)) * 100
        non_null_counts = df.notnull().sum()
        audit_df = pd.DataFrame({
            'Null Count': null_counts,
            'Null %': null_pcts,
            'Non-Null Count': non_null_counts
        })
        return audit_df
    except Exception as e:
        print(f"Error in missingness audit: {e}")
        return pd.DataFrame()

def detect_column_types(df):
    numeric_cols = []
    categorical_cols = []
    datetime_cols = []
    
    for col in df.columns:
        if pd.api.types.is_numeric_dtype(df[col]):
            numeric_cols.append(col)
        elif pd.api.types.is_datetime64_any_dtype(df[col]):
            datetime_cols.append(col)
        else:
            sample = df[col].dropna().head(100).astype(str)
            is_date = False
            if len(sample) > 0:
                try:
                    pd.to_datetime(sample, errors='raise')
                    is_date = True
                except:
                    pass
            if is_date:
                datetime_cols.append(col)
            else:
                categorical_cols.append(col)
    return numeric_cols, categorical_cols, datetime_cols

def safe_to_numeric(series):
    try:
        converted = pd.to_numeric(series, errors='coerce')
        if series.notnull().sum() > 0 and converted.notnull().sum() / series.notnull().sum() > 0.5:
            return converted, True
    except:
        pass
    return series, False

# B) Load + Validate
print(f"=== Loading dataset: {DATASET_NAME} ===")
df = safe_read_csv(DATA_DIR)

if not df.empty:
    actual_cols = list(df.columns)
    if EXACT_COLUMNS:
        missing_cols = [c for c in EXACT_COLUMNS if c not in actual_cols]
        extra_cols = [c for c in actual_cols if c not in EXACT_COLUMNS]
        if missing_cols or extra_cols:
            print("\n--- Column Validation Report ---")
            if missing_cols:
                print(f"Missing expected columns: {missing_cols}")
            if extra_cols:
                print(f"Extra columns found: {extra_cols}")
        else:
            print("\nColumn validation successful: Actual columns match expected columns.")
    else:
        print("\nNo expected columns provided for validation.")

    # C) Data Audit
    print("\n=== Data Audit ===")
    print(f"Shape: {df.shape}")
    print(f"Memory Usage (Deep): {df.memory_usage(deep=True).sum() / (1024**2):.2f} MB")
    print(f"Duplicate Rows Count: {df.duplicated().sum()}")

    print("\n--- Missingness Audit ---")
    missing_audit = audit_missingness(df)
    print(missing_audit)

    numeric_cols, categorical_cols, datetime_cols = detect_column_types(df)

    print("\n--- Cardinality Audit (Categorical/Object) ---")
    for col in categorical_cols:
        unique_cnt = df[col].nunique()
        top_val = df[col].mode().iloc[0] if not df[col].mode().empty else "N/A"
        print(f"Column '{col}': Unique Counts = {unique_cnt}, Top Value = {top_val}")

    print("\n--- Numeric Stability Audit ---")
    for col in numeric_cols:
        inf_count = np.isinf(df[col]).sum() if df[col].dtype in [np.float32, np.float64] else 0
        var = df[col].var()
        print(f"Column '{col}': Inf/-Inf Count = {inf_count}, Variance = {var:.4f}")
        if pd.notnull(var) and var < 1e-4:
            print(f"  -> Warning: Column '{col}' is near-constant (low variance).")

    print(f"\nDetected Datetime Column Candidates: {datetime_cols}")

    # D) ETL (Safe + Reversible)
    print("\n=== ETL Processing ===")
    df_clean = df.copy()

    cols_converted_to_numeric = []
    missing_tokens = ["", "NA", "N/A", "null", "None"]

    for col in df_clean.columns:
        if df_clean[col].dtype == 'object':
            df_clean[col] = df_clean[col].apply(lambda x: x.strip() if isinstance(x, str) else x)
            df_clean[col] = df_clean[col].replace(missing_tokens, np.nan)

    for col in datetime_cols:
        try:
            df_clean[col] = pd.to_datetime(df_clean[col], errors='coerce')
        except Exception as e:
            print(f"Error converting {col} to datetime: {e}")

    # Re-detect types after basic token handling
    numeric_cols, categorical_cols, datetime_cols = detect_column_types(df_clean)

    for col in categorical_cols:
        if df_clean[col].dtype == 'object':
            conv_series, success = safe_to_numeric(df_clean[col])
            if success:
                df_clean[col] = conv_series
                cols_converted_to_numeric.append(col)

    # Refresh column types if conversion happened
    if cols_converted_to_numeric:
        numeric_cols, categorical_cols, datetime_cols = detect_column_types(df_clean)

    # Missing value handling strategy
    for col in numeric_cols:
        if df_clean[col].isnull().any():
            df_clean[f"{col}__was_missing"] = df_clean[col].isnull().astype(int)
            median_val = df_clean[col].median()
            df_clean[col] = df_clean[col].fillna(median_val)

    for col in categorical_cols:
        if df_clean[col].isnull().any():
            df_clean[f"{col}__was_missing"] = df_clean[col].isnull().astype(int)
            df_clean[col] = df_clean[col].fillna("Missing")

    for col in datetime_cols:
        if df_clean[col].isnull().any():
            df_clean[f"{col}__was_missing"] = df_clean[col].isnull().astype(int)

    dup_count = df_clean.duplicated().sum()
    df_clean = df_clean.drop_duplicates(keep='first')

    # Outlier handling via winsorization into new columns
    for col in numeric_cols:
        non_null_samples = df_clean[col].dropna()
        if len(non_null_samples) > 0:
            try:
                q25 = non_null_samples.quantile(0.25)
                q75 = non_null_samples.quantile(0.75)
                iqr = q75 - q25
                if iqr > 0:
                    lower_bound = q25 - 1.5 * iqr
                    upper_bound = q75 + 1.5 * iqr
                    outliers = non_null_samples[(non_null_samples < lower_bound) | (non_null_samples > upper_bound)]
                    outlier_rate = len(outliers) / len(non_null_samples)
                    if outlier_rate > 0:
                        print(f"Column '{col}' outlier rate: {outlier_rate:.2%}")
                        q01 = non_null_samples.quantile(0.01)
                        q99 = non_null_samples.quantile(0.99)
                        df_clean[f"{col}__winsor"] = df_clean[col].clip(q01, q99)
            except Exception as e:
                print(f"Error handling outliers for {col}: {e}")

    # E) EDA
    print("\n=== EDA Summary ===")
    print("\n--- Numeric Distribution Summary Table ---")
    if numeric_cols:
        numeric_summary = df_clean[numeric_cols].agg(['mean', 'std', 'min', 'median', 'max', 'skew', 'kurt'])
        print(numeric_summary.T)

    print("\n--- Categorical Frequency Table (Top 5 Categories) ---")
    for col in categorical_cols:
        print(f"\nFrequency table for '{col}':")
        print(df_clean[col].value_counts().head(5))

    if len(numeric_cols) >= 2:
        print("\n--- Correlation Analysis ---")
        corr_cols = numeric_cols[:50]
        if len(numeric_cols) > 50:
            print(f"Note: Capping correlation computations to the first 50 numeric features.")
        
        corr_matrix = df_clean[corr_cols].corr(method='pearson')
        
        high_corr_pairs = []
        for i in range(len(corr_matrix.columns)):
            for j in range(i+1, len(corr_matrix.columns)):
                col1 = corr_matrix.columns[i]
                col2 = corr_matrix.columns[j]
                r_val = corr_matrix.iloc[i, j]
                if abs(r_val) >= 0.85:
                    high_corr_pairs.append((col1, col2, r_val))
                    
        if high_corr_pairs:
            print("Highly correlated pairs (|r| >= 0.85):")
            for c1, c2, r in high_corr_pairs:
                print(f"  {c1} & {c2} => r = {r:.4f}")
        else:
            print("No pairs found with |r| >= 0.85")

    if TARGET_COL and TARGET_COL in df_clean.columns:
        print(f"\n--- Target-Aware Analysis (Target: {TARGET_COL}) ---")
        if TARGET_COL in numeric_cols:
            print("Numeric target correlations:")
            corrs = df_clean[numeric_cols].corrwith(df_clean[TARGET_COL])
            print(corrs.sort_values(ascending=False))
        elif TARGET_COL in categorical_cols:
            print("Categorical target group stats:")
            print(df_clean[TARGET_COL].value_counts())

    # F) Visualization
    print("\n=== Generating Plots ===")
    try:
        null_pct = df.isnull().sum() / len(df) * 100
        null_pct = null_pct[null_pct > 0].sort_values(ascending=False).head(30)
        if not null_pct.empty:
            fig, ax = plt.subplots(figsize=(12, 5))
            sns.barplot(x=null_pct.values, y=null_pct.index, hue=null_pct.index, palette='viridis', legend=False, ax=ax)
            ax.set_title('Top Columns by Missing %')
            ax.set_xlabel('Missing %')
            plt.tight_layout()
            plt.savefig('missingness_chart.png')
            plt.close()
    except Exception as e:
        print(f"Error plotting missingness: {e}")

    try:
        plot_num_cols = [c for c in numeric_cols if not c.endswith('__was_missing') and not c.endswith('__winsor')][:12]
        if plot_num_cols:
            n_cols = 3
            n_rows = (len(plot_num_cols) + n_cols - 1) // n_cols
            fig, axes = plt.subplots(nrows=n_rows, ncols=n_cols, figsize=(15, n_rows * 3.5))
            if n_rows * n_cols == 1:
                axes = [axes]
            else:
                axes = axes.flatten()
            for idx, col in enumerate(plot_num_cols):
                sns.histplot(df_clean[col], kde=True, ax=axes[idx])
                axes[idx].set_title(f'Dist of {col}')
            for i in range(idx + 1, len(axes)):
                fig.delaxes(axes[i])
            plt.tight_layout()
            plt.savefig('numeric_histograms.png')
            plt.close()
    except Exception as e:
        print(f"Error plotting histograms: {e}")

    try:
        if plot_num_cols:
            n_cols = 3
            n_rows = (len(plot_num_cols) + n_cols - 1) // n_cols
            fig, axes = plt.subplots(nrows=n_rows, ncols=n_cols, figsize=(15, n_rows * 3.5))
            if n_rows * n_cols == 1:
                axes = [axes]
            else:
                axes = axes.flatten()
            for idx, col in enumerate(plot_num_cols):
                sns.boxplot(y=df_clean[col], ax=axes[idx])
                axes[idx].set_title(f'Boxplot of {col}')
            for i in range(idx + 1, len(axes)):
                fig.delaxes(axes[i])
            plt.tight_layout()
            plt.savefig('numeric_boxplots.png')
            plt.close()
    except Exception as e:
        print(f"Error plotting boxplots: {e}")

    try:
        plot_cat_cols = [c for c in categorical_cols if not c.endswith('__was_missing')][:6]
        if plot_cat_cols:
            n_cols = 2
            n_rows = (len(plot_cat_cols) + n_cols - 1) // n_cols
            fig, axes = plt.subplots(nrows=n_rows, ncols=n_cols, figsize=(15, n_rows * 4))
            if n_rows * n_cols == 1:
                axes = [axes]
            else:
                axes = axes.flatten()
            for idx, col in enumerate(plot_cat_cols):
                top_cats = df_clean[col].value_counts().head(10).index
                sub_df = df_clean[df_clean[col].isin(top_cats)]
                sns.countplot(data=sub_df, y=col, order=top_cats, hue=col, palette='muted', legend=False, ax=axes[idx])
                axes[idx].set_title(f'Top Values for {col}')
            for i in range(idx + 1, len(axes)):
                fig.delaxes(axes[i])
            plt.tight_layout()
            plt.savefig('categorical_countplots.png')
            plt.close()
    except Exception as e:
        print(f"Error plotting countplots: {e}")

    try:
        base_num_cols = [c for c in numeric_cols if not c.endswith('__was_missing') and not c.endswith('__winsor')][:20]
        if len(base_num_cols) >= 2:
            fig, ax = plt.subplots(figsize=(10, 8))
            sns.heatmap(df_clean[base_num_cols].corr(), annot=False, cmap='coolwarm', fmt=".2f", ax=ax)
            ax.set_title('Correlation Heatmap (Top 20 Numeric Features)')
            plt.tight_layout()
            plt.savefig('correlation_heatmap.png')
            plt.close()
    except Exception as e:
        print(f"Error plotting correlation heatmap: {e}")

    try:
        base_dt_cols = [c for c in datetime_cols if not c.endswith('__was_missing')]
        if base_dt_cols:
            dt_col = base_dt_cols[0]
            fig, ax = plt.subplots(figsize=(12, 5))
            df_clean.groupby(df_clean[dt_col].dt.year).size().plot(kind='bar', ax=ax)
            ax.set_title(f'Records over Time based on {dt_col}')
            ax.set_xlabel('Year')
            ax.set_ylabel('Count')
            plt.tight_layout()
            plt.savefig('datetime_trend.png')
            plt.close()
    except Exception as e:
        print(f"Error plotting datetime trend: {e}")

    # G) Lightweight Feature Engineering (Safe Extras)
    print("\n=== Feature Engineering ===")
    for col in datetime_cols:
        try:
            df_clean[f"{col}__year"] = df_clean[col].dt.year
            df_clean[f"{col}__month"] = df_clean[col].dt.month
            df_clean[f"{col}__dayofweek"] = df_clean[col].dt.dayofweek
            print(f"Derived year, month, dayofweek from '{col}'.")
        except Exception as e:
            print(f"Error engineering datetime features for {col}: {e}")

    for col in categorical_cols:
        try:
            df_clean[f"{col}__len"] = df_clean[col].astype(str).apply(len)
            df_clean[f"{col}__words"] = df_clean[col].astype(str).apply(lambda x: len(x.split()))
        except Exception as e:
            print(f"Error engineering text features for {col}: {e}")

    # H) Final Artifact Output
    print("\n=== Final Summary ===")
    print(f"Before Shape: {df.shape}")
    print(f"After Shape: {df_clean.shape}")

    print("\nData Quality Summary:")
    print(f"  - Duplicates removed: {dup_count}")
    print(f"  - Columns converted to numeric: {cols_converted_to_numeric}")
    print(f"  - Detected datetime columns: {datetime_cols}")
    print(f"  - Total missing values in clean dataset: {df_clean.isnull().sum().sum()}")

    print("\n--- Clean Dataset Head ---")
    print(df_clean.head())
else:
    print("Loaded DataFrame is empty.")


=== Loading dataset: afonsofernandescruz/2026-fifa-world-cup-historical-elo-ratings ===

Column validation successful: Actual columns match expected columns.

=== Data Audit ===
Shape: (4683, 23)
Memory Usage (Deep): 1.66 MB
Duplicate Rows Count: 0

--- Missingness Audit ---
                 Null Count  Null %  Non-Null Count
year                      0     0.0            4683
snapshot_date             0     0.0            4683
country                   0     0.0            4683
rank                      0     0.0            4683
country_code              0     0.0            4683
rating                    0     0.0            4683
rank_max                  0     0.0            4683
rating_max                0     0.0            4683
rank_avg                  0     0.0            4683
rating_avg                0     0.0            4683
rank_min                  0     0.0            4683
rating_min                0     0.0            4683
matches_total             0     0.0            4